In [22]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [23]:
df = pd.read_csv("drug200.csv")
df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY


In [ ]:
df=df.copy()

label_encoders={}

for col in df.columns:
    if df[col].dtype=='object':
        le=LabelEncoder()
        df[col]=le.fit_transform(df[col])
        label_encoders[col]=le

df.head()

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,0,0,0,25.355,4
1,47,1,1,0,13.093,2
2,47,1,1,0,10.114,2
3,28,0,2,0,7.798,3
4,61,0,1,0,18.043,4


In [ ]:
X=df.drop("Drug", axis=1).values
y=df["Drug"].values

In [ ]:
scaler=StandardScaler()
X=scaler.fit_transform(X)

In [ ]:
def one_hot(y, num_classes):
    onehot=np.zeros((len(y), num_classes))
    for i, val in enumerate(y):
        onehot[i, val]=1
    return onehot

num_classes=len(np.unique(y))
y_onehot=one_hot(y, num_classes)

In [ ]:
def softmax(z):
    exp=np.exp(z-np.max(z, axis=1, keepdims=True))
    return exp / np.sum(exp, axis=1, keepdims=True)

In [ ]:
class LogisticRegressionScratch:
    
    def __init__(self, lr=0.01, epochs=1000, reg=None, alpha=0.01, l1_ratio=0.5):
        self.lr=lr
        self.epochs=epochs
        self.reg=reg
        self.alpha=alpha
        self.l1_ratio=l1_ratio
        
    def fit(self, X, y):
        
        n_samples, n_features=X.shape
        n_classes=y.shape[1]
        
        self.W=np.zeros((n_features, n_classes))
        self.b=np.zeros((1, n_classes))
        
        for _ in range(self.epochs):
            
            linear=np.dot(X, self.W) + self.b
            y_pred=softmax(linear)
            
            error=y_pred - y
            
            dW=np.dot(X.T, error)/n_samples
            db=np.sum(error, axis=0, keepdims=True)/n_samples
            
            if self.reg=="ridge":
                dW += self.alpha * self.W
                
            elif self.reg=="lasso":
                dW += self.alpha * np.sign(self.W)
                
            elif self.reg=="elastic":
                dW += self.alpha * (self.l1_ratio*np.sign(self.W)+(1-self.l1_ratio)*self.W)
            
            self.W-=self.lr*dW
            self.b-=self.lr*db
            
    def predict(self, X):
        
        linear=np.dot(X, self.W) + self.b
        y_pred=softmax(linear)
        
        return np.argmax(y_pred, axis=1)

In [ ]:
kf=KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
def evaluate_model(model):

    acc_list=[]
    prec_list=[]
    rec_list=[]
    f1_list=[]
    
    for train_index, test_index in kf.split(X):
        
        X_train, X_test=X[train_index], X[test_index]
        y_train, y_test=y_onehot[train_index], y[test_index]
        
        model.fit(X_train, y_train)
        
        preds=model.predict(X_test)
        
        acc_list.append(accuracy_score(y_test, preds))
        prec_list.append(precision_score(y_test, preds, average='macro'))
        rec_list.append(recall_score(y_test, preds, average='macro'))
        f1_list.append(f1_score(y_test, preds, average='macro'))
        
    return np.mean(acc_list), np.mean(prec_list), np.mean(rec_list), np.mean(f1_list)

In [ ]:
model_none=LogisticRegressionScratch(reg=None)

acc, prec, rec, f1=evaluate_model(model_none)

print("No Regularization")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

No Regularization
Accuracy: 0.835
Precision: 0.6826281824470678
Recall: 0.6811846014322793
F1 Score: 0.6654166757591142


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaco

In [33]:
model_ridge=LogisticRegressionScratch(reg="ridge", alpha=0.01)

acc, prec, rec, f1=evaluate_model(model_ridge)

print("Ridge Regularization")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Ridge Regularization
Accuracy: 0.8300000000000001
Precision: 0.6914316012504866
Recall: 0.6678512680989461
F1 Score: 0.656170546512985


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [34]:
model_lasso=LogisticRegressionScratch(reg="lasso", alpha=0.01)

acc, prec, rec, f1=evaluate_model(model_lasso)

print("Lasso Regularization")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaco

Lasso Regularization
Accuracy: 0.82
Precision: 0.648033259310349
Recall: 0.6315375426087501
F1 Score: 0.6160287792876938


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [35]:
model_elastic=LogisticRegressionScratch(reg="elastic", alpha=0.01)

acc, prec, rec, f1=evaluate_model(model_elastic)

print("Elastic Net Regularization")
print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)

c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaco

Elastic Net Regularization
Accuracy: 0.835
Precision: 0.6581798267092385
Recall: 0.6595375426087501
F1 Score: 0.6419859595598146


c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [36]:
results=pd.DataFrame({
    "Model":["No Reg", "Ridge", "Lasso", "Elastic Net"],
    "Accuracy":[
        evaluate_model(LogisticRegressionScratch(reg=None))[0],
        evaluate_model(LogisticRegressionScratch(reg="ridge"))[0],
        evaluate_model(LogisticRegressionScratch(reg="lasso"))[0],
        evaluate_model(LogisticRegressionScratch(reg="elastic"))[0]
    ]
})

results

c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Suraj Shah\anaco

,Model,Accuracy
0,No Reg,0.835
1,Ridge,0.830
2,Lasso,0.820
3,Elastic Net,0.835
